# Part 1 — Experiment Overview

This notebook loads your antenna experiment data and gives you a bird's-eye view of what was collected.
By the end you will know:

- How many spots, runs, and unique RX stations are in the dataset
- What the SNR and distance distributions look like
- How spots are spread across bearing and time
- Whether data quality is sufficient for front/back analysis

**How it works:** When you clicked *Launch in JupyterLite* on the experiment page, the dashboard
exported your data as CSV files into this browser-based notebook environment.  Everything runs
locally in your browser — no server round-trips needed.

> Continue to **Part 2** for front/back analysis and **Part 3** for maps and space weather.

## 1 · Install interactive libraries

We use [Plotly](https://plotly.com/python/) for interactive charts (hover, zoom, pan) instead of
static matplotlib images.  Pyodide (the Python engine in this browser tab) can install it on the fly.

In [ ]:
import warnings, micropip
warnings.simplefilter('ignore', DeprecationWarning)
await micropip.install(['plotly', 'jinja2', 'nbformat'])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from io import StringIO
from pathlib import Path
print('Libraries ready ✓')

## 2 · Load experiment data

The handoff wrote up to four files into this workspace:

| File | Contents |
|------|----------|
| `experiment.csv` | Every WSPR spot collected across all runs |
| `run_ledger.csv` | One row per data-collection run (metadata, timing, role) |
| `qth_metadata.csv` | Your station QTH, coordinates, distance units |
| `space_weather.csv` | 7-day Kp / SFI / Bz history from NOAA |

The cell below finds and loads them.

In [ ]:
def _find(name):
    for p in [Path(name), Path('/'+name), Path('files/'+name), Path('/files/'+name)]:
        if p.exists(): return p
    return None

csv_path = _find('experiment.csv')
if csv_path is None:
    raise RuntimeError('experiment.csv not found — relaunch from the dashboard.')

# --- Main spots table ---
df = pd.read_csv(csv_path, dtype='string')
numeric_cols = ['snr','distance','distance_miles','distance_km',
                'antenna_azimuth','beacon_bin_azimuth','bearing','rx_lat','rx_lon']
for c in numeric_cols:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
for c in ['timestamp_utc','time']:
    if c in df.columns: df[c] = pd.to_datetime(df[c], errors='coerce', utc=True)
if 'snr' in df.columns:
    df['snr'] = np.round(df['snr']).astype('Int64').astype('float64')

# --- QTH metadata ---
distance_units = 'km'
qth_lat, qth_lon, qth_grid = None, None, None
meta_path = _find('qth_metadata.csv')
if meta_path:
    meta = pd.read_csv(meta_path)
    if len(meta):
        du = str(meta.iloc[0].get('distance_units','')).strip().lower()
        if du == 'mi': distance_units = 'mi'
        qth_lat = pd.to_numeric(meta.iloc[0].get('qth_lat'), errors='coerce')
        qth_lon = pd.to_numeric(meta.iloc[0].get('qth_lon'), errors='coerce')
        qth_grid = str(meta.iloc[0].get('qth','')).strip() or None

# Use profile distance units
if distance_units == 'mi' and 'distance_miles' in df.columns:
    df['distance'] = df['distance_miles']
elif distance_units == 'km' and 'distance_km' in df.columns:
    df['distance'] = df['distance_km']

# --- Run ledger ---
run_ledger = None
run_role_map = {}
rl_path = _find('run_ledger.csv')
if rl_path:
    run_ledger = pd.read_csv(rl_path)
    for c in ['kp','sfi','bz']:
        if c in run_ledger.columns: run_ledger[c] = pd.to_numeric(run_ledger[c], errors='coerce')
    if {'run_key','role'}.issubset(run_ledger.columns):
        tmp = run_ledger[['run_key','role']].dropna().astype(str)
        tmp['role'] = tmp['role'].str.lower()
        run_role_map = dict(zip(tmp['run_key'], tmp['role']))

# --- Space weather history ---
sw_df = None
sw_path = _find('space_weather.csv')
if sw_path:
    sw_df = pd.read_csv(sw_path)
    if 'timestamp' in sw_df.columns:
        sw_df['timestamp'] = pd.to_datetime(sw_df['timestamp'], errors='coerce', utc=True)
    for c in ['kp_index','solar_flux','sunspot_number','xray_flux','imf_bz']:
        if c in sw_df.columns: sw_df[c] = pd.to_numeric(sw_df[c], errors='coerce')

# --- Derive missing RX coordinates from Maidenhead grid ---
def grid_to_latlon(g):
    if not isinstance(g, str) or len(g) < 4: return (np.nan, np.nan)
    g = g.strip().upper()
    try:
        lon = -180 + (ord(g[0])-65)*20 + int(g[2])*2 + 1
        lat = -90 + (ord(g[1])-65)*10 + int(g[3]) + 0.5
        if len(g) >= 6 and g[4].isalpha() and g[5].isalpha():
            lon = -180 + (ord(g[0])-65)*20 + int(g[2])*2 + (ord(g[4])-65+0.5)*(2/24)
            lat = -90 + (ord(g[1])-65)*10 + int(g[3]) + (ord(g[5])-65+0.5)*(1/24)
        return (lat, lon)
    except Exception:
        return (np.nan, np.nan)

if {'rx_lat','rx_lon','rxGrid'}.issubset(df.columns):
    mask = df['rx_lat'].isna() | df['rx_lon'].isna()
    if mask.any():
        derived = df.loc[mask,'rxGrid'].apply(lambda g: grid_to_latlon(str(g)) if pd.notna(g) else (np.nan,np.nan))
        coords = pd.DataFrame(derived.tolist(), index=derived.index, columns=['_lat','_lon'])
        df.loc[mask & df['rx_lat'].isna(), 'rx_lat'] = coords['_lat']
        df.loc[mask & df['rx_lon'].isna(), 'rx_lon'] = coords['_lon']

print(f'Spots loaded: {len(df)}')
print(f'Distance units: {distance_units}')
print(f'QTH: {qth_grid}  ({qth_lat}, {qth_lon})')
print(f'Run ledger: {len(run_ledger) if run_ledger is not None else 0} runs')
print(f'Space weather: {len(sw_df) if sw_df is not None else 0} readings')
print(f'Columns: {list(df.columns)}')

## 3 · Summary statistics

A quick snapshot of the dataset: total spots, unique receivers, time span, and median values.

In [ ]:
time_col = 'time' if 'time' in df.columns else 'timestamp_utc'

summary = pd.DataFrame([{
    'Total spots': len(df),
    'Unique runs': df['run_key'].nunique() if 'run_key' in df.columns else '—',
    'Unique RX stations': df['rxCall'].nunique() if 'rxCall' in df.columns else '—',
    'Unique RX grids': df['rxGrid'].nunique() if 'rxGrid' in df.columns else '—',
    'SNR median (dB)': f"{df['snr'].median():.0f}" if 'snr' in df.columns and df['snr'].notna().any() else '—',
    'SNR range': f"{df['snr'].min():.0f} to {df['snr'].max():.0f}" if 'snr' in df.columns and df['snr'].notna().any() else '—',
    f'Distance median ({distance_units})': f"{df['distance'].median():.0f}" if 'distance' in df.columns and df['distance'].notna().any() else '—',
    'Time span (UTC)': f"{df[time_col].min()} → {df[time_col].max()}" if time_col in df.columns and df[time_col].notna().any() else '—',
}]).T.rename(columns={0:'Value'})

summary.style.set_properties(**{'text-align':'left'})

## 4 · Run ledger

Each data-collection run has metadata: timestamps, spot count, antenna heading, bin azimuth,
front/back role, and space weather conditions (Kp, SFI, Bz).  This table helps you verify
that runs were configured correctly and conditions were comparable.

In [ ]:
if run_ledger is not None and len(run_ledger):
    show_cols = [c for c in ['run_key','pair_id','start','stop','spot_count','unique_rx',
                             'front_azimuth','bin_azimuth','role','kp','sfi','bz'] if c in run_ledger.columns]
    styled = run_ledger[show_cols].style
    # Highlight front/back roles
    def role_color(val):
        if str(val).strip().lower() == 'front': return 'background-color: #dbeafe'
        if str(val).strip().lower() == 'back': return 'background-color: #fed7aa'
        return ''
    if 'role' in show_cols:
        styled = styled.map(role_color, subset=['role'])
    styled
else:
    print('No run ledger available.')

## 5 · SNR distribution

The SNR histogram shows how strong or weak the received signals were.  WSPR signals are
typically between −30 dB and +20 dB.  A healthy dataset has a bell-shaped distribution
centered somewhere in that range.

- **Hover** over bars to see exact counts.
- A long tail toward low SNR means many weak paths (common on higher bands or long distances).

In [ ]:
if 'snr' in df.columns and df['snr'].notna().any():
    fig = px.histogram(df, x='snr', nbins=40,
                       title='SNR Distribution (all spots)',
                       labels={'snr':'SNR (dB)','count':'Spots'},
                       color_discrete_sequence=['#3b82f6'])
    fig.add_vline(x=df['snr'].median(), line_dash='dash', line_color='#f97316',
                  annotation_text=f"median {df['snr'].median():.0f} dB")
    fig.update_layout(height=350, margin=dict(t=40,b=30))
    fig.show()
else:
    print('No SNR data available.')

## 6 · Distance distribution

How far away are the stations hearing your signal?  Distance affects SNR — longer paths
have more propagation loss but also tap different ionospheric layers.

In [ ]:
if 'distance' in df.columns and df['distance'].notna().any():
    fig = px.histogram(df, x='distance', nbins=40,
                       title=f'Path Distance Distribution ({distance_units})',
                       labels={'distance':f'Distance ({distance_units})','count':'Spots'},
                       color_discrete_sequence=['#8b5cf6'])
    fig.add_vline(x=df['distance'].median(), line_dash='dash', line_color='#f97316',
                  annotation_text=f"median {df['distance'].median():.0f}")
    fig.update_layout(height=350, margin=dict(t=40,b=30))
    fig.show()
else:
    print('No distance data.')

## 7 · Bearing distribution

This polar chart shows how many spots arrived from each compass direction.  Gaps indicate
directions with few or no RX stations — that matters for front/back comparison.

If you picked a bin azimuth, the front and back sectors are highlighted.

In [ ]:
if 'bearing' in df.columns and df['bearing'].notna().any():
    bin_edges = np.arange(0, 370, 10)
    df['bearing_bin'] = pd.cut(df['bearing'], bins=bin_edges, right=False,
                               labels=bin_edges[:-1].astype(int))
    counts = df.groupby('bearing_bin', observed=False).size().reset_index(name='spots')
    counts['bearing_bin'] = counts['bearing_bin'].astype(int)

    fig = px.bar_polar(counts, r='spots', theta='bearing_bin',
                       title='Spot Count by Bearing (10° bins)',
                       color='spots', color_continuous_scale='Viridis')
    fig.update_layout(height=800, polar=dict(angularaxis=dict(direction='clockwise', rotation=90)))
    fig.show()
else:
    print('No bearing data.')

## 8 · SNR vs Bearing scatter

Each dot is one WSPR spot.  The horizontal axis is compass bearing, the vertical axis is SNR.
Front-directed spots should ideally cluster higher than back-directed spots if the antenna
has directional gain.

Spots are colored by run — different runs appear in different colors so you can see
whether front and back runs are interleaved.

In [ ]:
if {'bearing','snr'}.issubset(df.columns) and len(df):
    color_col = 'run_key' if 'run_key' in df.columns and df['run_key'].nunique() > 1 else None
    fig = px.scatter(df, x='bearing', y='snr', color=color_col,
                     title='SNR vs Bearing (colored by run)',
                     labels={'bearing':'Bearing (°)','snr':'SNR (dB)'},
                     opacity=0.6, hover_data=['rxCall','distance'] if 'rxCall' in df.columns else None)
    fig.update_xaxes(range=[0,360], dtick=45)
    fig.update_layout(height=400, margin=dict(t=40,b=30))
    fig.show()
else:
    print('Missing bearing/snr columns.')

## 9 · Spots over time

This timeline shows when spots arrived.  Runs should be clearly separated in time,
and the ABAB interleaving pattern (front/back/front/back) should be visible.

Gaps between clusters are normal — WSPR transmits in 2-minute windows.

In [ ]:
time_col = 'time' if 'time' in df.columns and df['time'].notna().any() else 'timestamp_utc'
if time_col in df.columns and df[time_col].notna().any():
    plot_df = df.dropna(subset=[time_col]).copy()
    color_col = 'run_key' if 'run_key' in plot_df.columns and plot_df['run_key'].nunique() > 1 else None
    fig = px.scatter(plot_df, x=time_col, y='snr' if 'snr' in plot_df.columns else None,
                     color=color_col,
                     title='Spots Timeline',
                     labels={time_col:'Time (UTC)','snr':'SNR (dB)'},
                     opacity=0.6)
    fig.update_layout(height=350, margin=dict(t=40,b=30))
    fig.show()
else:
    print('No timestamp data.')

## 10 · Data quality check

The experiment protocol defines minimum thresholds for meaningful analysis:

| Metric | Minimum |
|--------|---------|
| Runs per orientation | 3 |
| Filtered spots per orientation | 30 |
| Distinct RX per orientation | 10 |
| Overlap stations (both front & back) | 6 |

The cell below checks these against your dataset.

In [ ]:
checks = []

# Runs per orientation
if run_ledger is not None and 'role' in run_ledger.columns:
    role_counts = run_ledger['role'].str.lower().value_counts()
    front_runs = int(role_counts.get('front', 0))
    back_runs = int(role_counts.get('back', 0))
    checks.append({'Check': 'Front runs', 'Value': front_runs, 'Minimum': 3,
                   'Status': '✓' if front_runs >= 3 else '✗'})
    checks.append({'Check': 'Back runs', 'Value': back_runs, 'Minimum': 3,
                   'Status': '✓' if back_runs >= 3 else '✗'})
else:
    checks.append({'Check': 'Run ledger', 'Value': 'missing', 'Minimum': '—', 'Status': '⚠'})

# Spots and RX per orientation (simple bearing-based split)
if {'bearing','beacon_bin_azimuth','snr'}.issubset(df.columns) and df['beacon_bin_azimuth'].notna().any():
    bin_az = df['beacon_bin_azimuth'].median()
    def ang_dist(a, b): return abs((a - b + 180) % 360 - 180)
    df['_qc_orient'] = df['bearing'].apply(lambda b: 'front' if ang_dist(b, bin_az) <= ang_dist(b, (bin_az+180)%360) else 'back')
    for orient in ['front','back']:
        sub = df[df['_qc_orient'] == orient]
        n_spots = len(sub)
        n_rx = sub['rxCall'].nunique() if 'rxCall' in sub.columns else 0
        checks.append({'Check': f'{orient.title()} spots', 'Value': n_spots, 'Minimum': 30,
                       'Status': '✓' if n_spots >= 30 else '✗'})
        checks.append({'Check': f'{orient.title()} unique RX', 'Value': n_rx, 'Minimum': 10,
                       'Status': '✓' if n_rx >= 10 else '✗'})

    # Overlap
    if 'rxCall' in df.columns:
        front_rx = set(df.loc[df['_qc_orient']=='front','rxCall'].dropna())
        back_rx = set(df.loc[df['_qc_orient']=='back','rxCall'].dropna())
        overlap = len(front_rx & back_rx)
        checks.append({'Check': 'Overlap RX stations', 'Value': overlap, 'Minimum': 6,
                       'Status': '✓' if overlap >= 6 else '✗'})
    df.drop(columns=['_qc_orient'], inplace=True, errors='ignore')

qc = pd.DataFrame(checks)
def qc_color(val):
    if val == '✓': return 'color: #16a34a; font-weight: bold'
    if val == '✗': return 'color: #dc2626; font-weight: bold'
    return 'color: #d97706'
qc.style.map(qc_color, subset=['Status'])

## 11 · Sample data preview

A scrollable preview of the first 50 rows — useful for sanity-checking column values.

In [ ]:
preview_cols = [c for c in ['time','rxCall','rxGrid','band','snr','distance','bearing',
                            'antenna_azimuth','beacon_bin_azimuth','run_key','pair_role']
                if c in df.columns]
df[preview_cols].head(50)

---

**Next:** Open **`02_front_back_analysis.ipynb`** to compute front/back estimates
using three progressively stronger methods — simple sector medians, station-paired
deltas, and **HDBSCAN cluster analysis** with Hodges-Lehmann estimators and bootstrap
confidence intervals.  Then open **`03_maps_and_weather.ipynb`** for interactive RX
station maps, space weather correlation, and the ionospheric confidence penalty.